# Sesi 9 · Statistik Deskriptif dan Analisis Pelanggan

**Kursus:** Data Analysis dengan BigQuery, Python & BI
**Nama:** *tulis namamu di sini*
**Tanggal:** 28 November 2026

Manajemen bertanya: **"Siapa pelanggan terbaik kita, dan apakah mereka kembali berbelanja?"** Hari ini kita menjawabnya dengan:

1. **Statistik deskriptif**: rata-rata, median, sebaran, dan kenapa rata-rata bisa menipu.
2. **Repeat-purchase rate**: berapa persen pelanggan yang berbelanja lebih dari sekali.
3. **Segmentasi RFM**: mengelompokkan pelanggan berdasarkan *Recency*, *Frequency*, dan *Monetary*.

> Simpan salinan dulu: **File → Save a copy in Drive**.

> **VERSI INSTRUKTUR · KUNCI JAWABAN.** Jangan dibagikan ke peserta. Semua sel latihan (kerjakan bersama dan latihan mandiri) sudah terisi; angka di jawaban teks ditandai `[...]` dan diisi dari output saat dijalankan.

## 0. Persiapan

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from google.cloud import bigquery

PROJECT_ID = "isi-project-id-kamu"   # <-- GANTI dengan Project ID-mu
client = bigquery.Client(project=PROJECT_ID)
pd.set_option("display.float_format", "{:,.2f}".format)

def jalankan_query(sql):
    """Menjalankan query SQL di BigQuery dan mengembalikan hasilnya sebagai DataFrame."""
    return client.query(sql).to_dataframe()

WARNA_UTAMA = "#2a78d6"
WARNA_SOROT = "#eb6834"
WARNA_TEKS = "#5f6368"
plt.rcParams.update({
    "figure.figsize": (10, 5), "font.size": 11,
    "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e8eaed", "axes.axisbelow": True,
})
os.makedirs("images", exist_ok=True)

### Data: satu baris per pesanan

Untuk analisis pelanggan, satuan yang kita butuhkan adalah **pesanan**, bukan barang. Kita jumlahkan barang per pesanan langsung di SQL.

In [ ]:
SQL_PESANAN = """
SELECT
  oi.order_id,
  oi.user_id,
  DATE(MIN(oi.created_at)) AS order_date,
  COUNT(*) AS jumlah_barang,
  SUM(oi.sale_price) AS nilai_pesanan,
  ANY_VALUE(u.country) AS country,
  ANY_VALUE(u.gender) AS gender,
  ANY_VALUE(u.traffic_source) AS traffic_source
FROM `bigquery-public-data.thelook_ecommerce.order_items` AS oi
JOIN `bigquery-public-data.thelook_ecommerce.users` AS u ON oi.user_id = u.id
WHERE oi.status NOT IN ('Cancelled', 'Returned')
  AND DATE(oi.created_at) < DATE_TRUNC(CURRENT_DATE(), MONTH)
GROUP BY oi.order_id, oi.user_id
"""
pesanan = jalankan_query(SQL_PESANAN)
pesanan["order_date"] = pd.to_datetime(pesanan["order_date"])
print(pesanan.shape)
pesanan.head()

`ANY_VALUE(kolom)` mengambil satu nilai dari kelompok. Aman dipakai di sini karena negara, gender, dan traffic source sama untuk semua barang dalam satu pesanan.

# Bagian A · Statistik deskriptif

| Ukuran | Arti | pandas |
| --- | --- | --- |
| Rata-rata (*mean*) | Jumlah ÷ banyaknya data | `.mean()` |
| Median | Nilai tengah setelah diurutkan | `.median()` |
| Modus | Nilai yang paling sering muncul | `.mode()` |
| Simpangan baku (*std*) | Seberapa jauh data menyebar dari rata-rata | `.std()` |
| Persentil / kuartil | Nilai di posisi tertentu (misalnya 90% data di bawahnya) | `.quantile(0.9)` |

In [ ]:
pesanan["nilai_pesanan"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])

In [ ]:
rata = pesanan["nilai_pesanan"].mean()
median = pesanan["nilai_pesanan"].median()
print(f"Rata-rata nilai pesanan : {rata:,.2f} USD")
print(f"Median nilai pesanan    : {median:,.2f} USD")

### Kenapa rata-rata lebih besar dari median?

Sebagian kecil pesanan bernilai sangat besar dan "menarik" rata-rata ke atas. Sebaran seperti ini disebut **miring ke kanan** (*right-skewed*), dan sangat umum di data penjualan, pendapatan, atau harga rumah.

**Aturan praktis:** kalau data miring, laporkan **median** sebagai "nilai tipikal", dan sebutkan rata-ratanya sebagai pelengkap.

In [ ]:
batas_tampil = pesanan["nilai_pesanan"].quantile(0.99)   # potong 1% teratas supaya grafik terbaca

fig, ax = plt.subplots()
ax.hist(pesanan.loc[pesanan["nilai_pesanan"] <= batas_tampil, "nilai_pesanan"],
        bins=50, color=WARNA_UTAMA, edgecolor="white", linewidth=0.5)
ax.axvline(median, color=WARNA_SOROT, linewidth=2, label=f"Median {median:,.0f} USD")
ax.axvline(rata, color=WARNA_TEKS, linewidth=2, linestyle="--", label=f"Rata-rata {rata:,.0f} USD")
ax.set_title("Kebanyakan pesanan bernilai kecil; sedikit pesanan besar menarik rata-rata ke atas")
ax.set_xlabel("Nilai pesanan (USD), 1% teratas tidak ditampilkan")
ax.set_ylabel("Jumlah pesanan")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Membandingkan kelompok

In [ ]:
# Nilai pesanan per traffic source: rata-rata vs median, dan jumlah pesanan
(pesanan.groupby("traffic_source")["nilai_pesanan"]
        .agg(["count", "mean", "median"])
        .sort_values("count", ascending=False))

In [ ]:
# Jumlah barang per pesanan: proporsi (%)
pesanan["jumlah_barang"].value_counts(normalize=True).sort_index() * 100

# Bagian B · Repeat-purchase rate

**Repeat-purchase rate** = persentase pelanggan yang berbelanja **2 kali atau lebih**, dari semua pelanggan yang pernah berbelanja.

Ini salah satu indikator kesehatan bisnis e-commerce: mendatangkan pelanggan baru biasanya jauh lebih mahal daripada membuat pelanggan lama kembali.

In [ ]:
pesanan_per_pelanggan = pesanan.groupby("user_id")["order_id"].nunique()

total_pelanggan = len(pesanan_per_pelanggan)
pelanggan_berulang = (pesanan_per_pelanggan >= 2).sum()
repeat_rate = pelanggan_berulang / total_pelanggan * 100

print(f"Pelanggan yang pernah berbelanja : {total_pelanggan:,}")
print(f"Berbelanja 2 kali atau lebih     : {pelanggan_berulang:,}")
print(f"Repeat-purchase rate             : {repeat_rate:.1f}%")

In [ ]:
# Sebaran jumlah pesanan per pelanggan
sebaran = pesanan_per_pelanggan.value_counts().sort_index()
sebaran_persen = (sebaran / sebaran.sum() * 100).round(1)
pd.DataFrame({"jumlah_pelanggan": sebaran, "persen": sebaran_persen})

In [ ]:
# Apakah repeat rate berbeda antar traffic source?
per_pelanggan = pesanan.groupby("user_id").agg(
    jumlah_pesanan=("order_id", "nunique"),
    traffic_source=("traffic_source", "first"),
)
per_pelanggan["berulang"] = per_pelanggan["jumlah_pesanan"] >= 2
(per_pelanggan.groupby("traffic_source")["berulang"].mean() * 100).sort_values(ascending=False).round(1)

`.mean()` pada kolom True/False menghasilkan **proporsi True**, karena True dihitung 1 dan False dihitung 0. Trik ini sangat sering dipakai.

# Bagian C · Segmentasi RFM

RFM mengelompokkan pelanggan berdasarkan tiga pertanyaan:

| Huruf | Pertanyaan | Cara hitung |
| --- | --- | --- |
| **R**ecency | Kapan terakhir berbelanja? | Hari sejak pesanan terakhir (makin kecil makin baik) |
| **F**requency | Seberapa sering? | Jumlah pesanan |
| **M**onetary | Berapa banyak uang yang dibelanjakan? | Total nilai pesanan |

## C1. Menghitung R, F, M per pelanggan

In [ ]:
# Tanggal acuan: sehari setelah pesanan terakhir di data
tanggal_acuan = pesanan["order_date"].max() + pd.Timedelta(days=1)
print("Tanggal acuan:", tanggal_acuan.date())

rfm = pesanan.groupby("user_id").agg(
    terakhir_belanja=("order_date", "max"),
    frequency=("order_id", "nunique"),
    monetary=("nilai_pesanan", "sum"),
)
rfm["recency"] = (tanggal_acuan - rfm["terakhir_belanja"]).dt.days
rfm = rfm[["recency", "frequency", "monetary"]]
rfm.head()

In [ ]:
rfm.describe()

## C2. Memberi skor

- **R** → skor 1–4 berdasarkan kuartil: 25% pelanggan yang paling baru berbelanja mendapat skor 4. Kita pakai `pd.qcut`, yang membagi data menjadi kelompok berukuran sama.
- **F** → sebagian besar pelanggan hanya berbelanja sekali, jadi kuartil tidak cocok (banyak nilai kembar). Kita pakai aturan tetap: 1 pesanan = 1, 2 pesanan = 2, 3+ pesanan = 3.
- **M** → skor 1–4 berdasarkan kuartil: 25% pelanggan dengan belanja terbesar mendapat skor 4.

In [ ]:
# .rank(method="first") memberi peringkat unik, supaya nilai kembar tidak membuat qcut error
rfm["skor_r"] = pd.qcut(rfm["recency"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)  # recency kecil = skor besar
rfm["skor_f"] = rfm["frequency"].clip(upper=3)                                                    # 3 atau lebih -> 3
rfm["skor_m"] = pd.qcut(rfm["monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm.head()

## C3. Dari skor menjadi segmen

Skor saja sulit dipahami manajemen. Kita terjemahkan menjadi **nama segmen** yang menjelaskan tindakan:

| Segmen | Aturan | Artinya | Tindakan |
| --- | --- | --- | --- |
| **Champions** | R ≥ 3 dan F ≥ 2 | Baru belanja dan sudah berulang | Program loyalitas, akses awal produk baru |
| **Pelanggan Baru** | R ≥ 3 dan F = 1 | Baru belanja, baru sekali | Dorong pembelian kedua: voucher, rekomendasi |
| **Berisiko Pergi** | R ≤ 2 dan F ≥ 2 | Dulu sering, sekarang lama tidak belanja | Kampanye *win-back* yang personal |
| **Tidak Aktif** | R ≤ 2 dan F = 1 | Sekali belanja, lalu hilang | Biaya rendah: email otomatis saja |

In [ ]:
def segmen_rfm(baris):
    """Mengembalikan nama segmen RFM dari skor R dan F satu pelanggan."""
    if baris["skor_r"] >= 3 and baris["skor_f"] >= 2:
        return "Champions"
    elif baris["skor_r"] >= 3:
        return "Pelanggan Baru"
    elif baris["skor_f"] >= 2:
        return "Berisiko Pergi"
    else:
        return "Tidak Aktif"

rfm["segmen"] = rfm.apply(segmen_rfm, axis=1)   # axis=1: function diterapkan per baris
rfm["segmen"].value_counts()

In [ ]:
ringkasan = rfm.groupby("segmen").agg(
    jumlah_pelanggan=("recency", "count"),
    recency_median=("recency", "median"),
    frequency_rata=("frequency", "mean"),
    monetary_rata=("monetary", "mean"),
    total_belanja=("monetary", "sum"),
)
ringkasan["porsi_pelanggan_%"] = ringkasan["jumlah_pelanggan"] / ringkasan["jumlah_pelanggan"].sum() * 100
ringkasan["porsi_pendapatan_%"] = ringkasan["total_belanja"] / ringkasan["total_belanja"].sum() * 100
urutan = ["Champions", "Pelanggan Baru", "Berisiko Pergi", "Tidak Aktif"]
ringkasan = ringkasan.reindex(urutan)
ringkasan

## C4. Grafik: porsi pelanggan vs porsi pendapatan

Pesan yang ingin disampaikan: **apakah segmen kecil menyumbang pendapatan besar?** Dua batang per segmen, satu sumbu (persen), sehingga mudah dibandingkan.

In [ ]:
import numpy as np

posisi = np.arange(len(ringkasan))
lebar = 0.38

fig, ax = plt.subplots()
ax.bar(posisi - lebar / 2, ringkasan["porsi_pelanggan_%"], lebar, color=WARNA_UTAMA, label="% pelanggan")
ax.bar(posisi + lebar / 2, ringkasan["porsi_pendapatan_%"], lebar, color=WARNA_SOROT, label="% pendapatan")

for i, (p, d) in enumerate(zip(ringkasan["porsi_pelanggan_%"], ringkasan["porsi_pendapatan_%"])):
    ax.text(i - lebar / 2, p + 0.8, f"{p:.0f}%", ha="center", color=WARNA_TEKS, fontsize=10)
    ax.text(i + lebar / 2, d + 0.8, f"{d:.0f}%", ha="center", color=WARNA_TEKS, fontsize=10)

champ = ringkasan.loc["Champions"]
ax.set_title(f"Champions: {champ['porsi_pelanggan_%']:.0f}% pelanggan, "
             f"{champ['porsi_pendapatan_%']:.0f}% pendapatan")
ax.set_xticks(posisi, ringkasan.index)
ax.set_ylabel("Persen (%)")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("images/04_segmen_rfm.png", dpi=150, bbox_inches="tight")
plt.show()

## C5. Siapa saja di setiap segmen?

In [ ]:
# Gabungkan segmen ke data pesanan untuk melihat profilnya
profil = pesanan.merge(rfm[["segmen"]], left_on="user_id", right_index=True)

# Traffic source di setiap segmen (persen per baris)
pd.crosstab(profil["segmen"], profil["traffic_source"], normalize="index").reindex(urutan) * 100

`merge` = JOIN di pandas. `left_on="user_id", right_index=True` artinya kolom `user_id` di `pesanan` dicocokkan dengan indeks `rfm` (yang berisi `user_id`). `pd.crosstab` membuat tabel silang, mirip pivot table.

## Kerjakan bersama (±10 menit)

Instruktur mengerjakannya langkah demi langkah di layar, dan kamu mengetik bersamaan di sel di bawah.

**B1.** Hitung repeat-purchase rate **per negara** untuk 5 negara dengan pelanggan terbanyak.

In [ ]:
# B1
pp = pesanan.groupby("user_id").agg(jumlah_pesanan=("order_id", "nunique"),
                                    country=("country", "first"))
pp["berulang"] = pp["jumlah_pesanan"] >= 2
top5 = pp["country"].value_counts().head(5).index
(pp[pp["country"].isin(top5)].groupby("country")["berulang"].mean() * 100).round(1).sort_values(ascending=False)

**B2.** Berapa median `monetary` pelanggan **Champions** dibanding **Pelanggan Baru**? Berapa kali lipat?

In [ ]:
# B2
median_m = rfm.groupby("segmen")["monetary"].median()
print(median_m)
print(f"{median_m['Champions'] / median_m['Pelanggan Baru']:.1f} kali lipat")

## Unduh grafik

In [ ]:
from google.colab import files
files.download("images/04_segmen_rfm.png")

# Tiga temuan untuk manajemen

**Ditulis bersama di kelas.** Instruktur menunjukkan angkanya dan mengajukan pertanyaan pemandu; kamu menuliskannya dengan kata-katamu sendiri di sel di bawah.

Tulis **3 temuan** dari sesi 7–9 yang menurutmu paling penting bagi manajemen TheLook. Setiap temuan berisi:

1. **Apa** yang kamu temukan (dengan angka).
2. **Kenapa** itu penting bagi bisnis.
3. **Bukti**: nama grafik atau tabel di notebook-mu.

Contoh format:

> **Pelanggan yang kembali berbelanja masih sedikit.** Hanya X% pelanggan yang berbelanja lebih dari sekali. Artinya sebagian besar pendapatan bergantung pada pelanggan baru, yang biaya akuisisinya mahal. *(Bukti: Bagian B, tabel sebaran jumlah pesanan.)*

Lalu kita tulis **1 rekomendasi**, dan bersama-sama menyalin temuan + rekomendasi ke `README.md` repo **`thelook-python-analysis`** (pakai template README Python dari repo kursus), serta mengunggah notebook ini ke `notebooks/09_customer_analysis.ipynb` dan grafiknya ke `images/`.

*Contoh jawaban. Isi angka `[...]` dari output notebook hari itu.*

**Temuan 1: Sebagian besar pelanggan hanya berbelanja sekali.** Repeat-purchase rate hanya [X]%. Pertumbuhan pendapatan sangat bergantung pada pelanggan baru, yang biaya akuisisinya lebih mahal daripada mempertahankan pelanggan lama. *(Bukti: Bagian B, sebaran jumlah pesanan per pelanggan.)*

**Temuan 2: Segmen Champions kecil tetapi bernilai besar.** Champions hanya [X]% pelanggan tetapi menyumbang [Y]% pendapatan, dengan median belanja [N] kali Pelanggan Baru. Program loyalitas untuk segmen ini melindungi porsi pendapatan yang besar. *(Bukti: Bagian C4, grafik segmen RFM.)*

**Temuan 3: Nilai pesanan tipikal lebih kecil dari kesan rata-rata.** Median nilai pesanan [X] USD, sedangkan rata-ratanya [Y] USD karena sebagian kecil pesanan besar. Target dan promosi sebaiknya memakai median sebagai acuan "pesanan biasa". *(Bukti: Bagian A, histogram nilai pesanan.)*

**Rekomendasi:** Jalankan program pembelian kedua untuk segmen Pelanggan Baru (misalnya voucher 30 hari setelah pesanan pertama), dan ukur keberhasilannya dari kenaikan repeat-purchase rate dari [X]% ke target [Y]%.

In [ ]:
# (Opsional) Ruang untuk analisis tambahan yang mendukung temuanmu